In [2]:
import pandas as pd

# あなたが作ったCSVを読み込む
df = pd.read_csv("npb_2024_main_fourth_batter_stats.csv")

# 数値化
num_cols = ["本塁打", "OBP", "SLG", "OPS", "打点", "試合", "打席", "打数", "安打", "四球", "死球", "三振", "併殺打"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# zスコア標準化
def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

df["z_HR"] = zscore(df["本塁打"])
df["z_OBP"] = zscore(df["OBP"])
df["z_SLG"] = zscore(df["SLG"])
df["z_OPS"] = zscore(df["OPS"])

# スコア作成
df["長打型スコア"] = 0.5 * df["z_HR"] + 0.3 * df["z_SLG"] + 0.2 * df["z_OPS"]
df["総合型スコア"] = 0.4 * df["z_OBP"] + 0.3 * df["z_SLG"] + 0.3 * df["z_OPS"]

# 分類
df["4番タイプ"] = df.apply(
    lambda row: "長打型" if row["長打型スコア"] > row["総合型スコア"] else "総合型",
    axis=1
)

# 見やすい形に整理
result = df[[
    "球団", "選手名", "4番出場回数",
    "本塁打", "OBP", "SLG", "OPS", "打点",
    "長打型スコア", "総合型スコア", "4番タイプ"
]].copy()

# スコアを見やすく丸める
result["長打型スコア"] = result["長打型スコア"].round(3)
result["総合型スコア"] = result["総合型スコア"].round(3)

# 保存
result.to_csv("npb_2024_main_fourth_batter_types.csv", index=False, encoding="utf-8-sig")

print(result.sort_values("長打型スコア", ascending=False))
print("\n保存完了: npb_2024_main_fourth_batter_types.csv")

        球団     選手名  4番出場回数  本塁打    OBP    SLG    OPS  打点  長打型スコア  総合型スコア 4番タイプ
3     ヤクルト   村上 宗隆     131   33  0.379  0.472  0.851  86   1.219   1.292   総合型
2   ソフトバンク   山川 穂高     143   34  0.318  0.484  0.801  99   1.181  -0.159   長打型
6       巨人   岡本 和真     143   27  0.362  0.501  0.863  83   1.114   1.158   総合型
0     DeNA    牧 秀悟      79   23  0.346  0.491  0.837  74   0.769   0.640   長打型
5       中日   細川 成也      86   23  0.368  0.478  0.846  67   0.724   1.072   総合型
4      ロッテ      ソト     116   21  0.330  0.450  0.780  88   0.252  -0.201   長打型
8     日本ハム  マルティネス      79   13  0.336  0.405  0.740  57  -0.543  -0.522   総合型
1    オリックス    森 友哉      62    9  0.368  0.415  0.783  46  -0.556   0.412   総合型
9       楽天   浅村 栄斗      76   14  0.346  0.382  0.728  60  -0.661  -0.500   総合型
11      阪神   大山 悠輔      90   14  0.338  0.383  0.721  68  -0.678  -0.696   長打型
10      西武   佐藤 龍世      32    7  0.330  0.390  0.720  34  -1.001  -0.829   総合型
7       広島   小園 海斗      71    2  0.322  0.330  0.651